# Кейс 6. Метрические методы регрессии

## Импорты

Импортируем всё необходимое. numpy для вычислений, matplotlib для графиков, из sklearn берём только датасеты, стандартизацию и разбивку. Все алгоритмы регрессии пишем сами.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


## Данные

Используем два реальных датасета и два синтетических.

**Diabetes** — 442 объекта, 10 признаков. Компактный датасет для регрессии, удобен для отладки и LOO.

**California Housing** — 20640 объектов, 8 признаков. Задача регрессии с признаками разного масштаба. Для LOO берём подвыборку, иначе считается слишком долго.

**Синтетика y = sin(x) + eps** — одномерная задача для визуализации прогноза при разных h.

**Синтетика с выбросами** — та же функция, но с искусственно добавленными выбросами для исследования LOWESS.

In [ ]:
diabetes = load_diabetes()
X_diab, y_diab = diabetes.data, diabetes.target
print(f"Diabetes: {X_diab.shape}")

housing = fetch_california_housing()
X_hous, y_hous = housing.data, housing.target
print(f"California Housing: {X_hous.shape}")

# Одномерная синтетическая задача
n_synth = 150
X_synth = np.sort(np.random.uniform(-3*np.pi, 3*np.pi, n_synth))
y_synth = np.sin(X_synth) + np.random.normal(0, 0.3, n_synth)

# Синтетика с выбросами
y_outlier = y_synth.copy()
outlier_idx = np.random.choice(n_synth, size=15, replace=False)
y_outlier[outlier_idx] += np.random.choice([-4, 4], size=15)

print(f"Синтетика: {X_synth.shape}, добавлено выбросов: {len(outlier_idx)}")


## Ядра

Те же четыре ядра что в кейсе 5. Задают насколько сильно влияет объект в зависимости от расстояния до него.

In [ ]:
def kernel_uniform(r):
    return (np.abs(r) <= 1).astype(float)

def kernel_triangular(r):
    return np.maximum(0, 1 - np.abs(r))

def kernel_epanechnikov(r):
    return np.maximum(0, 0.75*(1 - r**2))

def kernel_gaussian(r):
    return np.exp(-0.5 * r**2)

KERNELS = {
    'Uniform':      kernel_uniform,
    'Triangular':   kernel_triangular,
    'Epanechnikov': kernel_epanechnikov,
    'Gaussian':     kernel_gaussian,
}

r = np.linspace(-2.5, 2.5, 300)
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, K in KERNELS.items():
    ax.plot(r, K(r), label=name, linewidth=1.8)
ax.set_xlabel('r'); ax.set_ylabel('K(r)')
ax.set_title('Ядра')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Регрессия Надарая-Ватсона, фиксированное окно

Оценка регрессии в точке x — взвешенное среднее ответов:

$$a(x) = \frac{\sum_{i=1}^{\ell} y_i\, K\!\left(\frac{\rho(x,x_i)}{h}\right)}{\sum_{i=1}^{\ell} K\!\left(\frac{\rho(x,x_i)}{h}\right)}$$

Если знаменатель близок к нулю — нет объектов в окне, возвращаем среднее по выборке.

In [ ]:
def nw_fixed(x, X_train, y_train, h, kernel):
    dists = np.abs(x - X_train) if X_train.ndim == 1 else np.linalg.norm(x - X_train, axis=1)
    w = kernel(dists / h)
    denom = w.sum()
    return (w * y_train).sum() / denom if denom > 1e-12 else y_train.mean()

def nw_fixed_batch(X_test, X_train, y_train, h, kernel):
    return np.array([nw_fixed(x, X_train, y_train, h, kernel) for x in X_test])

# Проверка
pred = nw_fixed(0.0, X_synth, y_synth, h=1.0, kernel=kernel_gaussian)
print(f"Прогноз в x=0: {pred:.4f}, истинное sin(0)=0.0")


## Регрессия Надарая-Ватсона, переменное окно

Ширина окна адаптируется к локальной плотности: h(x) = расстояние до (k+1)-го ближайшего соседа.

In [ ]:
def nw_variable(x, X_train, y_train, k, kernel):
    dists = np.abs(x - X_train) if X_train.ndim == 1 else np.linalg.norm(x - X_train, axis=1)
    h = max(np.sort(dists)[k], 1e-9)
    w = kernel(dists / h)
    denom = w.sum()
    return (w * y_train).sum() / denom if denom > 1e-12 else y_train.mean()

def nw_variable_batch(X_test, X_train, y_train, k, kernel):
    return np.array([nw_variable(x, X_train, y_train, k, kernel) for x in X_test])


## LOWESS — робастная регрессия

Итеративно пересчитываем веса объектов gamma_i. Выбросы получают малый вес и на следующей итерации влияют меньше.

На каждой итерации:
1. Считаем LOO-прогнозы через NW с текущими весами gamma_i
2. Обновляем: gamma_i = Ke(|a_i - y_i| / (6 * median(eps)))

Ke — бисквитное ядро.

In [ ]:
def kernel_bisquare(r):
    return np.where(np.abs(r) < 1, (1 - r**2)**2, 0.0)

def lowess(X_train, y_train, h, kernel, n_iter=3):
    n = len(y_train)
    gamma = np.ones(n)

    for _ in range(n_iter):
        a_loo = np.zeros(n)
        for i in range(n):
            dists = np.abs(X_train[i] - X_train) if X_train.ndim == 1                     else np.linalg.norm(X_train[i] - X_train, axis=1)
            w = kernel(dists / h) * gamma
            w[i] = 0.0
            denom = w.sum()
            a_loo[i] = (w * y_train).sum() / denom if denom > 1e-12 else y_train.mean()

        eps = np.abs(a_loo - y_train)
        med = np.median(eps)
        if med < 1e-12:
            break
        gamma = kernel_bisquare(eps / (6 * med))

    return gamma, a_loo

def lowess_predict(X_test, X_train, y_train, gamma, h, kernel):
    def predict_one(x):
        dists = np.abs(x - X_train) if X_train.ndim == 1                 else np.linalg.norm(x - X_train, axis=1)
        w = kernel(dists / h) * gamma
        denom = w.sum()
        return (w * y_train).sum() / denom if denom > 1e-12 else y_train.mean()
    return np.array([predict_one(x) for x in X_test])


## Метрики качества

MAE, RMSE и R² для оценки регрессии.

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1 - ss_res / ss_tot


## Подбор h по LOO

Для каждого h считаем LOO MSE на синтетической выборке.

In [ ]:
def loo_nw_fixed(X, y, h_list, kernel):
    n = len(y)
    errors = []
    for h in h_list:
        err = sum(
            (nw_fixed(X[i], np.delete(X, i), np.delete(y, i), h, kernel) - y[i])**2
            for i in range(n)
        )
        errors.append(err / n)
    return np.array(errors)

h_list = np.linspace(0.1, 3.0, 30)
loo_h = loo_nw_fixed(X_synth, y_synth, h_list, kernel_gaussian)
best_h = h_list[np.argmin(loo_h)]
print(f"Оптимальное h* = {best_h:.3f}, LOO MSE = {loo_h.min():.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h_list, loo_h, 'o-', color='#2c7bb6', linewidth=1.5, markersize=4)
ax.axvline(best_h, color='red', linestyle='--', linewidth=1, label=f'h* = {best_h:.2f}')
ax.set_xlabel('h'); ax.set_ylabel('LOO MSE')
ax.set_title('LOO-ошибка vs h (синтетика, ядро Gaussian)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Подбор k для переменного окна по LOO

In [ ]:
def loo_nw_variable(X, y, k_list, kernel):
    n = len(y)
    errors = []
    for k in k_list:
        err = sum(
            (nw_variable(X[i], np.delete(X, i), np.delete(y, i), k, kernel) - y[i])**2
            for i in range(n)
        )
        errors.append(err / n)
    return np.array(errors)

k_list = np.arange(1, 31)
loo_k = loo_nw_variable(X_synth, y_synth, k_list, kernel_gaussian)
best_k = k_list[np.argmin(loo_k)]
print(f"Оптимальное k* = {best_k}, LOO MSE = {loo_k.min():.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_list, loo_k, 's-', color='#d7191c', linewidth=1.5, markersize=4)
ax.axvline(best_k, color='navy', linestyle='--', linewidth=1, label=f'k* = {best_k}')
ax.set_xlabel('k'); ax.set_ylabel('LOO MSE')
ax.set_title('LOO-ошибка vs k (переменное окно, синтетика, ядро Gaussian)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## Сравнение ядер

Сравниваем LOO MSE для четырёх ядер при оптимальном h. Хотим проверить насколько выбор ядра влияет на качество.

In [ ]:
print(f"{'Ядро':<15} {'LOO MSE':>10} {'h*':>8}")
print("-" * 35)
kernel_results = {}
for name, K in KERNELS.items():
    loo = loo_nw_fixed(X_synth, y_synth, h_list, K)
    bh = h_list[np.argmin(loo)]
    kernel_results[name] = (loo.min(), bh)
    print(f"{name:<15} {loo.min():>10.4f} {bh:>8.3f}")


## Графики на синтетической задаче

Строим прогнозы при разных h чтобы увидеть как ширина окна влияет на сглаживание. Маленькое h — переобучение, большое h — избыточное сглаживание.

In [ ]:
X_grid = np.linspace(X_synth.min(), X_synth.max(), 300)
h_vals = [0.2, best_h, 2.5]
colors = ['#e41a1c', '#377eb8', '#4daf4a']

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, h_val, col in zip(axes, h_vals, colors):
    y_pred = nw_fixed_batch(X_grid, X_synth, y_synth, h=h_val, kernel=kernel_gaussian)
    ax.scatter(X_synth, y_synth, s=15, alpha=0.5, color='gray', label='Наблюдения')
    ax.plot(X_grid, np.sin(X_grid), 'k--', linewidth=1.2, label='sin(x)')
    ax.plot(X_grid, y_pred, color=col, linewidth=2, label=f'h = {h_val:.2f}')
    ax.set_title(f'h = {h_val:.2f}')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Надарая-Ватсон при разных h (ядро Gaussian)', y=1.02)
plt.tight_layout(); plt.show()


## LOWESS vs обычное сглаживание при выбросах

Добавляем искусственные выбросы и сравниваем NW и LOWESS. LOWESS итеративно снижает вес выбросов.

In [ ]:
y_nw_out = nw_fixed_batch(X_grid, X_synth, y_outlier, best_h, kernel_gaussian)
gamma_final, _ = lowess(X_synth, y_outlier, best_h, kernel_gaussian, n_iter=3)
y_low_out = lowess_predict(X_grid, X_synth, y_outlier, gamma_final, best_h, kernel_gaussian)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, y_pred, title, col in zip(
    axes,
    [y_nw_out, y_low_out],
    ['Надарая-Ватсон (обычный)', 'LOWESS (робастный)'],
    ['#e41a1c', '#377eb8']
):
    ax.scatter(X_synth, y_outlier, s=15, alpha=0.4, color='gray', label='Данные с выбросами')
    ax.plot(X_grid, np.sin(X_grid), 'k--', linewidth=1.2, label='sin(x)')
    ax.plot(X_grid, y_pred, color=col, linewidth=2, label=title)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Сравнение NW и LOWESS при наличии выбросов', y=1.02)
plt.tight_layout(); plt.show()

y_true_g = np.sin(X_grid)
print(f"NW:     MAE={mae(y_true_g, y_nw_out):.4f}, RMSE={rmse(y_true_g, y_nw_out):.4f}, R2={r2(y_true_g, y_nw_out):.4f}")
print(f"LOWESS: MAE={mae(y_true_g, y_low_out):.4f}, RMSE={rmse(y_true_g, y_low_out):.4f}, R2={r2(y_true_g, y_low_out):.4f}")


## Веса gamma_i после стабилизации LOWESS

Смотрим какие объекты получили малый вес — это и есть выбросы.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sc = ax.scatter(X_synth, y_outlier, c=gamma_final, cmap='RdYlGn',
                s=40, edgecolors='k', linewidths=0.3, vmin=0, vmax=1)
plt.colorbar(sc, ax=ax, label='Вес gamma_i')
ax.plot(X_grid, np.sin(X_grid), 'k--', linewidth=1.2, label='sin(x)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Веса gamma_i (зелёный = большой вес, красный = выброс)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Объектов с весом < 0.1: {(gamma_final < 0.1).sum()}")
print(f"Реально добавлено выбросов: {len(outlier_idx)}")


## Распределение ошибок на объектах

Сравниваем гистограммы LOO-ошибок NW и LOWESS на данных с выбросами.

In [ ]:
a_nw_loo = np.array([
    nw_fixed(X_synth[i], np.delete(X_synth, i), np.delete(y_outlier, i),
             best_h, kernel_gaussian)
    for i in range(len(X_synth))
])
_, a_low_loo = lowess(X_synth, y_outlier, best_h, kernel_gaussian, n_iter=3)

eps_nw = np.abs(a_nw_loo - y_outlier)
eps_low = np.abs(a_low_loo - y_outlier)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(eps_nw, bins=25, color='#e41a1c', alpha=0.7, edgecolor='k')
axes[0].set_title('Ошибки — NW')
axes[0].set_xlabel('|a_i - y_i|'); axes[0].set_ylabel('Число объектов')
axes[0].grid(True, alpha=0.3)

axes[1].hist(eps_low, bins=25, color='#377eb8', alpha=0.7, edgecolor='k')
axes[1].set_title('Ошибки — LOWESS')
axes[1].set_xlabel('|a_i - y_i|'); axes[1].set_ylabel('Число объектов')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Распределение ошибок на объектах (данные с выбросами)')
plt.tight_layout(); plt.show()


## Качество на реальных датасетах

MAE, RMSE, R2 для NW фикс, NW перем и LOWESS на Diabetes и California Housing.

In [ ]:
datasets = {
    'Diabetes':      (X_diab, y_diab),
    'Housing (500)': (X_hous[:500], y_hous[:500]),
}

print(f"{'Датасет':<18} {'Метод':<18} {'MAE':>8} {'RMSE':>8} {'R2':>8}")
print("-" * 62)

for dname, (Xd, yd) in datasets.items():
    Xd_sc = StandardScaler().fit_transform(Xd)
    X_tr, X_te, y_tr, y_te = train_test_split(Xd_sc, yd, test_size=0.3, random_state=42)

    # Подбираем h по LOO на подвыборке
    h_grid = np.linspace(0.3, 3.0, 15)
    n_loo = min(80, len(y_tr))
    idx_loo = np.random.choice(len(y_tr), n_loo, replace=False)
    X_l, y_l = X_tr[idx_loo], y_tr[idx_loo]
    loo_d = [
        sum((nw_fixed(X_l[i], np.delete(X_l,i,0), np.delete(y_l,i), hh, kernel_gaussian) - y_l[i])**2
            for i in range(n_loo)) / n_loo
        for hh in h_grid
    ]
    bh_d = h_grid[np.argmin(loo_d)]
    bk_d = 10

    y_nw_f = nw_fixed_batch(X_te, X_tr, y_tr, bh_d, kernel_gaussian)
    y_nw_v = nw_variable_batch(X_te, X_tr, y_tr, bk_d, kernel_gaussian)
    gam_d, _ = lowess(X_tr, y_tr, bh_d, kernel_gaussian, n_iter=2)
    y_low_d = lowess_predict(X_te, X_tr, y_tr, gam_d, bh_d, kernel_gaussian)

    for mname, yp in [('NW фикс', y_nw_f), ('NW перем', y_nw_v), ('LOWESS', y_low_d)]:
        print(f"{dname:<18} {mname:<18} {mae(y_te,yp):>8.3f} {rmse(y_te,yp):>8.3f} {r2(y_te,yp):>8.3f}")
    print()


## При каком уровне загрязнения LOWESS выигрывает

Постепенно увеличиваем долю выбросов и смотрим при каком проценте LOWESS начинает давать меньший RMSE.

In [ ]:
outlier_fractions = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
rmse_nw_list, rmse_low_list = [], []

for frac in outlier_fractions:
    y_noisy = y_synth.copy()
    n_out = int(frac * n_synth)
    if n_out > 0:
        idx_out = np.random.choice(n_synth, n_out, replace=False)
        y_noisy[idx_out] += np.random.choice([-4, 4], size=n_out)

    y_nw_p = nw_fixed_batch(X_grid, X_synth, y_noisy, best_h, kernel_gaussian)
    gam, _ = lowess(X_synth, y_noisy, best_h, kernel_gaussian, n_iter=3)
    y_low_p = lowess_predict(X_grid, X_synth, y_noisy, gam, best_h, kernel_gaussian)

    y_true_g = np.sin(X_grid)
    rmse_nw_list.append(rmse(y_true_g, y_nw_p))
    rmse_low_list.append(rmse(y_true_g, y_low_p))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([f*100 for f in outlier_fractions], rmse_nw_list,
        'o-', color='#e41a1c', linewidth=1.8, markersize=6, label='NW фикс')
ax.plot([f*100 for f in outlier_fractions], rmse_low_list,
        's-', color='#377eb8', linewidth=1.8, markersize=6, label='LOWESS')
ax.set_xlabel('Доля выбросов, %')
ax.set_ylabel('RMSE')
ax.set_title('RMSE в зависимости от доли выбросов')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for frac, r_nw, r_low in zip(outlier_fractions, rmse_nw_list, rmse_low_list):
    winner = 'LOWESS' if r_low < r_nw else 'NW'
    print(f"Выбросов {frac*100:.0f}%: NW={r_nw:.4f}, LOWESS={r_low:.4f}  -> {winner}")


## Фиксированное vs переменное окно

Сравниваем на синтетике.

In [ ]:
y_nw_f_s = nw_fixed_batch(X_grid, X_synth, y_synth, best_h, kernel_gaussian)
y_nw_v_s = nw_variable_batch(X_grid, X_synth, y_synth, best_k, kernel_gaussian)
y_true_g = np.sin(X_grid)

print(f"NW фикс:  RMSE={rmse(y_true_g, y_nw_f_s):.4f}, R2={r2(y_true_g, y_nw_f_s):.4f}")
print(f"NW перем: RMSE={rmse(y_true_g, y_nw_v_s):.4f}, R2={r2(y_true_g, y_nw_v_s):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, y_pred, title, col in zip(
    axes,
    [y_nw_f_s, y_nw_v_s],
    [f'NW фикс (h={best_h:.2f})', f'NW перем (k={best_k})'],
    ['#e41a1c', '#377eb8']
):
    ax.scatter(X_synth, y_synth, s=15, alpha=0.4, color='gray', label='Наблюдения')
    ax.plot(X_grid, y_true_g, 'k--', linewidth=1.2, label='sin(x)')
    ax.plot(X_grid, y_pred, color=col, linewidth=2, label=title)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

fig.suptitle('Фиксированное vs переменное окно (синтетика)')
plt.tight_layout(); plt.show()


## Что важнее — ядро или ширина окна

Сравниваем разброс LOO MSE по ядрам и по значениям h.

In [ ]:
print("При оптимальном h для каждого ядра:")
best_errors = []
for name, K in KERNELS.items():
    loo = loo_nw_fixed(X_synth, y_synth, h_list, K)
    best_errors.append(loo.min())
    print(f"  {name:<15}: LOO MSE = {loo.min():.4f}  (h* = {h_list[np.argmin(loo)]:.2f})")

h_bad = 0.3
print(f"\nПри фиксированном плохом h={h_bad}:")
bad_errors = []
for name, K in KERNELS.items():
    loo = loo_nw_fixed(X_synth, y_synth, [h_bad], K)
    bad_errors.append(loo[0])
    print(f"  {name:<15}: LOO MSE = {loo[0]:.4f}")

print(f"\nРазброс по ядрам при opt h:  {max(best_errors) - min(best_errors):.4f}")
print(f"Разброс по ядрам при bad h:   {max(bad_errors) - min(bad_errors):.4f}")
print(f"Разброс по h (Gaussian):      {max(loo_h) - min(loo_h):.4f}")
print("\nВывод: разброс по h >> разброс по ядрам -> ширина окна важнее ядра")


## Выводы

1. **Ширина окна важнее выбора ядра.** Разброс LOO MSE по четырём ядрам при оптимальном h минимален. Переход от оптимального h к плохому увеличивает ошибку на порядок. Выбор ядра второстепенен.

2. **Переменное окно выигрывает при неравномерной плотности данных.** На синтетике с равномерной выборкой разница небольшая. На реальных данных переменное окно стабильнее — адаптируется к локальной плотности и не требует глобального подбора h.

3. **LOWESS выигрывает у обычного сглаживания при доле выбросов от 5-10%.** При чистых данных оба метода дают схожий результат. При загрязнении свыше 10% LOWESS заметно устойчивее: выбросы получают малый вес и почти не влияют на прогноз.

4. **Веса gamma_i надёжно идентифицируют выбросы.** После 3 итераций объекты с gamma_i < 0.1 почти точно совпадают с реально добавленными выбросами. Полезно не только для робастности но и как инструмент детекции аномалий.
